# Kalimantan Fire Situation Monitor — Phase 2
## Konfirmasi Area Terbakar & Penilaian Tingkat Keparahan (Burned-Area & Severity)

**Tujuan Utama:** Mengonfirmasi apakah klaster anomali termal (titik panas) prioritas dari **Phase 1** memiliki bukti fisik perubahan tutupan lahan yang konsisten dengan area bekas terbakar (*burned area*), serta mengukur tingkat keparahan ekologisnya (*burn severity*) menggunakan citra satelit optis multispektral beresolusi tinggi.

---

### 🛰️ Mengapa Phase 2 Diperlukan?
1. **Titik Panas (Hotspot VIIRS) $\neq$ Luas Hutan Terbakar:**  
   Sensor termal VIIRS (375m) hanya mendeteksi adanya radiasi panas saat api sedang menyala. Sensor termal tidak dapat mengukur berapa hektare vegetasi yang benar-benar hangus atau seberapa parah kerusakan kanopi pohon.
2. **Konfirmasi Menggunakan Citra Optis (Sentinel-2 & Landsat):**  
   Dengan membandingkan kondisi vegetasi sebelum kebakaran (*pre-fire*) dan sesudah kebakaran (*post-fire*), kita dapat memetakan batas fisik area terbakar (*burned perimeter*) dan mengklasifikasikan tingkat keparahannya.

---

### 🔄 Alur Kerja Analisis (Workflow):
1. **Memuat Klaster Phase 1:** Membaca data klaster spasial DBSCAN 7 hari dari hasil analisis Phase 1.
2. **Skor Prioritas Klaster (*Cluster Priority Score*):** Memeringkat klaster berdasarkan jumlah titik api, daya radiasi (FRP), tingkat keyakinan (*confidence*), dan durasi aktifnya.
3. **Pemilihan Top 20 Klaster:** Memilih 20 klaster paling signifikan untuk dianalisis secara mendalam.
4. **Pengambilan Citra Optis Dual-Sensor:** Menggunakan **Sentinel-2 L2A (10–20m)** sebagai sumber utama, dengan *fallback* otomatis ke **Landsat 8/9 L2 (30m)** jika awan terlalu tebal.
5. **Penyaringan Awan (*Cloud Masking*) & Komposit Bebas Awan:** Menerapkan baseline pra-kebakaran 30 hari (dapat diperluas ke 60 hari) dan pasca-kebakaran.
6. **Kalkulasi NBR & dNBR:** Menghitung indeks spektral *Normalized Burn Ratio* (NBR) dan *Differenced NBR* (dNBR).
7. **Klasifikasi Severitas Standar USGS:** Mengelompokkan dampak kebakaran ke dalam 5 kelas tingkat keparahan (*Key & Benson, 2006*).
8. **Delineasi Poligon & Ekspor GeoTIFF:** Menghasilkan poligon batas area terbakar (GeoJSON) dan raster dNBR (GeoTIFF) untuk GIS.
9. **Validasi Silang MODIS:** Membandingkan hasil delineasi dengan produk global MODIS MCD64A1 (500m).


## 02 — Pengaturan Lingkungan & Dependensi
Menginstal dan memuat pustaka (*library*) Python untuk analisis spasial dan komputasi Google Earth Engine.


In [ ]:
# Instalasi paket yang diperlukan di lingkungan Google Colab
!pip install -q earthengine-api geemap geopandas folium matplotlib rasterio scikit-learn requests

import os
import sys
import json
import math
import requests
from datetime import datetime, timezone, timedelta

import ee
import geemap
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
from shapely.geometry import shape, Point, Polygon, MultiPolygon, box as shapely_box

print("Pustaka geospatial dan analisis data berhasil dimuat:")
print(f" - Earth Engine API : {ee.__version__}")
print(f" - GeoPandas         : {gpd.__version__}")
print(f" - Geemap           : {geemap.__version__}")
print(f" - Pandas           : {pd.__version__}")


## 03 — Konfigurasi Parameter Analisis
Seluruh parameter, ambang batas (*threshold*), dan jalur penyimpanan diatur pada bagian ini.


In [ ]:
# =============================================================================
# KONFIGURASI PARAMETER (PHASE 2)
# =============================================================================

# Project ID Google Earth Engine
EE_PROJECT_ID = 'riset-banjarnegara'

# Koleksi Citra Satelit Optis (Terverifikasi di Katalog GEE)
S2_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'      # Sentinel-2 Level-2A (Permukaan / BOA)
LANDSAT9_COLLECTION = 'LANDSAT/LC09/C02/T1_L2'     # Landsat 9 Collection 2 Level-2
LANDSAT8_COLLECTION = 'LANDSAT/LC08/C02/T1_L2'     # Landsat 8 Collection 2 Level-2
MODIS_BURNED_COLLECTION = 'MODIS/061/MCD64A1'      # Referensi MODIS Burned Area Bulanan (500m)

# Batas Wilayah Administrasi Indonesia
ADMIN_L1_COLLECTION = 'WM/geoLab/geoBoundaries/600/ADM1'
ADMIN_L2_COLLECTION = 'WM/geoLab/geoBoundaries/600/ADM2'

# Pengaturan Fokus Analisis
TOP_N_CLUSTERS = 20           # Menganalisis 20 klaster paling prioritas
CLUSTER_BUFFER_KM = 5.0       # Radius area analisis di sekitar sentroid klaster (5 km)

# Bobot Skor Prioritas Klaster (Bukan Skor Risiko Bencana, Melainkan Prioritas Analisis)
W_COUNT = 0.35    # Bobot jumlah titik deteksi aktif
W_FRP = 0.30      # Bobot daya radiasi termal maksimum (FRP)
W_CONF = 0.15     # Bobot rata-rata tingkat keyakinan (confidence)
W_PERSIST = 0.20  # Bobot durasi aktif kebakaran (hari)

# Jendela Waktu Komposit Optis
BASELINE_DAYS = 30            # Baseline pra-kebakaran default (30 hari sebelum api terdeteksi)
FALLBACK_BASELINE_DAYS = 60   # Perluasan baseline jika tutupan awan > 50%
POST_FIRE_DAYS = 30           # Jendela waktu pasca-kebakaran
CLOUD_THRESHOLD_PCT = 50.0    # Ambang batas awan untuk memicu fallback

# Ambang Batas Klasifikasi Tingkat Keparahan Kebakaran (USGS / Key & Benson, 2006)
DNBR_BURNED_THRESHOLD = 0.10  # Batas minimum dNBR untuk dikategorikan sebagai area terbakar
SEVERITY_CLASSES = {
    'Unburned': {'min': -2.0, 'max': 0.10, 'color': '#006400', 'label': 'Tidak Terbakar / Pemulihan Vegetasi'},
    'Low': {'min': 0.10, 'max': 0.27, 'color': '#7FFF00', 'label': 'Keparahan Rendah (Low)'},
    'Moderate-Low': {'min': 0.27, 'max': 0.44, 'color': '#FFD700', 'label': 'Keparahan Sedang-Rendah (Mod-Low)'},
    'Moderate-High': {'min': 0.44, 'max': 0.66, 'color': '#FF8C00', 'label': 'Keparahan Sedang-Tinggi (Mod-High)'},
    'High': {'min': 0.66, 'max': 2.00, 'color': '#FF0000', 'label': 'Keparahan Tinggi (High)'}
}

# Direktori Penyimpanan Hasil (Lokal & Google Drive)
RUN_DATE_STR = datetime.now(timezone.utc).strftime('%Y-%m-%d')
LOCAL_OUTPUT_DIR = f'./outputs/{RUN_DATE_STR}/phase2'
DRIVE_RUN_PATH = f'/content/drive/MyDrive/Kalimantan-Fire-Monitor/{RUN_DATE_STR}/phase2'

# Jalur Pencarian Input dari Phase 1
PHASE1_INPUT_DIRS = [
    f'/content/drive/MyDrive/Kalimantan-Fire-Monitor/{RUN_DATE_STR}',
    f'/content/drive/MyDrive/Kalimantan-Fire-Monitor/2026-08-20',
    f'./export/{RUN_DATE_STR}',
    f'./export/2026-08-20',
    f'./outputs/{RUN_DATE_STR}',
    f'./outputs/2026-08-20'
]

print("Konfigurasi Phase 2 berhasil diinisialisasi:")
print(f" - Target Analisis : Top {TOP_N_CLUSTERS} Klaster Prioritas (Buffer: {CLUSTER_BUFFER_KM} km)")
print(f" - Baseline Pra-Api: {BASELINE_DAYS} hari (Fallback: {FALLBACK_BASELINE_DAYS} hari)")
print(f" - Sensor Utama    : Sentinel-2 L2A Harmonized (Resolusi 10–20m)")
print(f" - Sensor Cadangan : Landsat 8/9 C2 L2 (Resolusi 30m)")


## 04 — Otentikasi Google Earth Engine
Menghubungkan sesi Google Colab dengan Google Earth Engine menggunakan Google Cloud Project `riset-banjarnegara`.


In [ ]:
try:
    ee.Initialize(project=EE_PROJECT_ID)
    print(f"Google Earth Engine berhasil terhubung dengan project: '{EE_PROJECT_ID}'")
except Exception as e:
    print(f"Koneksi otomatis memerlukan otentikasi ({e}). Membuka jendela otentikasi...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
    print(f"Google Earth Engine berhasil diautentikasi dan terhubung dengan project: '{EE_PROJECT_ID}'")


## 05 — Memuat Data Klaster Hasil Phase 1
Mencari dan memuat file statistik klaster 7 hari (`cluster_statistics_7d.csv`) dari folder hasil ekspor Phase 1.


In [ ]:
def find_phase1_cluster_file():
    '''
    Mencari file cluster_statistics_7d.csv pada direktori lokal maupun Google Drive.
    '''
    for candidate in PHASE1_INPUT_DIRS:
        p1 = os.path.join(candidate, 'reports', 'cluster_statistics_7d.csv')
        if os.path.exists(p1):
            return p1
        p2 = os.path.join(candidate, 'cluster_statistics_7d.csv')
        if os.path.exists(p2):
            return p2
    return None

cluster_file = find_phase1_cluster_file()

if cluster_file and os.path.exists(cluster_file):
    print(f"File statistik klaster Phase 1 ditemukan di: {cluster_file}")
    df_raw_clusters = pd.read_csv(cluster_file)
else:
    print("Catatan: File lokal Phase 1 tidak ditemukan. Menggunakan data representatif klaster Agustus 2026...")
    # Data cadangan representatif dari eksekusi Phase 1
    df_raw_clusters = pd.DataFrame([
        {'cluster_id': 327, 'n_detections': 216, 'centroid_lat': 2.274, 'centroid_lon': 117.902, 'max_frp': 60.74, 'avg_frp': 9.31, 'avg_confidence': 1.82, 'first_date': '2026-08-14', 'last_date': '2026-08-20', 'province': 'East Kalimantan', 'regency': 'Berau'},
        {'cluster_id': 58,  'n_detections': 117, 'centroid_lat': -1.938, 'centroid_lon': 110.248, 'max_frp': 37.29, 'avg_frp': 7.40, 'avg_confidence': 1.25, 'first_date': '2026-08-15', 'last_date': '2026-08-19', 'province': 'West Kalimantan', 'regency': 'Ketapang'},
        {'cluster_id': 318, 'n_detections': 113, 'centroid_lat': 0.211, 'centroid_lon': 109.794, 'max_frp': 88.94, 'avg_frp': 15.51, 'avg_confidence': 1.60, 'first_date': '2026-08-16', 'last_date': '2026-08-20', 'province': 'West Kalimantan', 'regency': 'Landak'},
        {'cluster_id': 243, 'n_detections': 103, 'centroid_lat': 0.157, 'centroid_lon': 110.486, 'max_frp': 41.93, 'avg_frp': 7.86, 'avg_confidence': 1.45, 'first_date': '2026-08-14', 'last_date': '2026-08-20', 'province': 'West Kalimantan', 'regency': 'Sanggau'},
        {'cluster_id': 299, 'n_detections': 95,  'centroid_lat': 0.086, 'centroid_lon': 110.951, 'max_frp': 77.58, 'avg_frp': 10.99, 'avg_confidence': 1.55, 'first_date': '2026-08-15', 'last_date': '2026-08-20', 'province': 'West Kalimantan', 'regency': 'Sekadau'},
        {'cluster_id': 314, 'n_detections': 91,  'centroid_lat': 1.845, 'centroid_lon': 117.421, 'max_frp': 29.26, 'avg_frp': 5.74, 'avg_confidence': 1.20, 'first_date': '2026-08-15', 'last_date': '2026-08-20', 'province': 'East Kalimantan', 'regency': 'Kutai Timur'},
        {'cluster_id': 256, 'n_detections': 90,  'centroid_lat': 2.890, 'centroid_lon': 116.890, 'max_frp': 91.00, 'avg_frp': 19.66, 'avg_confidence': 1.70, 'first_date': '2026-08-16', 'last_date': '2026-08-20', 'province': 'North Kalimantan', 'regency': 'Bulungan'},
        {'cluster_id': 317, 'n_detections': 67,  'centroid_lat': 3.120, 'centroid_lon': 116.120, 'max_frp': 31.51, 'avg_frp': 11.48, 'avg_confidence': 1.40, 'first_date': '2026-08-14', 'last_date': '2026-08-19', 'province': 'North Kalimantan', 'regency': 'Malinau'},
        {'cluster_id': 16,  'n_detections': 54,  'centroid_lat': -2.150, 'centroid_lon': 112.980, 'max_frp': 26.53, 'avg_frp': 3.24, 'avg_confidence': 1.10, 'first_date': '2026-08-13', 'last_date': '2026-08-18', 'province': 'Central Kalimantan', 'regency': 'Kotawaringin Timur'},
        {'cluster_id': 298, 'n_detections': 48,  'centroid_lat': 0.450, 'centroid_lon': 112.850, 'max_frp': 50.91, 'avg_frp': 11.34, 'avg_confidence': 1.50, 'first_date': '2026-08-15', 'last_date': '2026-08-20', 'province': 'West Kalimantan', 'regency': 'Kapuas Hulu'}
    ])

print(f"Total {len(df_raw_clusters)} klaster berhasil dimuat untuk analisis Phase 2.")
print(df_raw_clusters[['cluster_id', 'n_detections', 'max_frp', 'avg_frp', 'province', 'regency']].head(10))


## 06 — Perhitungan Skor Prioritas & Pemilihan Top 20 Klaster
Memeringkat seluruh klaster menggunakan formula **Cluster Priority Score**:
$$\text{Score} = w_{\text{count}} \cdot \tilde{N} + w_{\text{frp}} \cdot \widetilde{\text{FRP}}_{\text{max}} + w_{\text{conf}} \cdot \tilde{C}_{\text{avg}} + w_{\text{persist}} \cdot \tilde{D}$$
*Catatan: Skor ini adalah metrik prioritas pemrosesan citra optis, bukan indeks bahaya/risiko bencana.*


In [ ]:
def calculate_cluster_priority_scores(df):
    '''
    Menghitung skor prioritas multi-kriteria untuk memilih klaster terpenting.
    '''
    df = df.copy()
    
    # Menghitung durasi aktivitas kebakaran (hari)
    df['first_dt'] = pd.to_datetime(df['first_date'])
    df['last_dt'] = pd.to_datetime(df['last_date'])
    df['duration_days'] = (df['last_dt'] - df['first_dt']).dt.days + 1
    
    # Fungsi normalisasi Min-Max [0, 1]
    def min_max(series):
        min_v = series.min()
        max_v = series.max()
        if max_v == min_v:
            return pd.Series(1.0, index=series.index)
        return (series - min_v) / (max_v - min_v)
    
    norm_n = min_max(df['n_detections'])
    norm_frp = min_max(df['max_frp'])
    norm_conf = min_max(df['avg_confidence'])
    norm_dur = min_max(df['duration_days'])
    
    df['priority_score'] = (
        W_COUNT * norm_n +
        W_FRP * norm_frp +
        W_CONF * norm_conf +
        W_PERSIST * norm_dur
    ).round(4)
    
    df = df.sort_values('priority_score', ascending=False).reset_index(drop=True)
    df['priority_rank'] = df.index + 1
    return df

df_prioritized = calculate_cluster_priority_scores(df_raw_clusters)
top20_clusters = df_prioritized.head(TOP_N_CLUSTERS).copy()

print("=" * 80)
print(f"TOP {len(top20_clusters)} KLASTER PRIORITAS UNTUK ANALISIS CITRA OPTIS")
print("=" * 80)
display_cols = ['priority_rank', 'cluster_id', 'priority_score', 'n_detections', 'max_frp',
                'duration_days', 'province', 'regency', 'centroid_lat', 'centroid_lon']
print(top20_clusters[display_cols].to_string(index=False))

# Grafik Skor Prioritas Klaster Teratas
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(top20_clusters)), top20_clusters['priority_score'], color='#E64A19', edgecolor='#333333')
ax.set_xticks(range(len(top20_clusters)))
ax.set_xticklabels([f"#{r['cluster_id']}\n({r['regency'][:8]})" for _, r in top20_clusters.iterrows()], rotation=45, ha='right')
ax.set_title(f'Peringkat Top {len(top20_clusters)} Klaster Prioritas Analisis Optis', fontweight='bold')
ax.set_ylabel('Skor Prioritas (0–1)')
for idx, val in enumerate(top20_clusters['priority_score']):
    ax.text(idx, val + 0.02, f"{val:.2f}", ha='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.show()


## 07 — Fungsi Masking Awan & Perhitungan Indeks Spektral (NBR)
Mendefinisikan algoritma pembersihan awan (*cloud masking*) dan bayangan awan untuk Sentinel-2 L2A (`QA60` & `SCL`) serta Landsat 8/9 (`QA_PIXEL`).

### Formulasi NBR (Normalized Burn Ratio):
$$\text{NBR} = \frac{\rho_{\text{NIR}} - \rho_{\text{SWIR2}}}{\rho_{\text{NIR}} + \rho_{\text{SWIR2}}}$$
* Sentinel-2: Menggunakan Band `B8A` (NIR 865nm) dan Band `B12` (SWIR2 2190nm).
* Landsat 8/9: Menggunakan Band `SR_B5` (NIR 865nm) dan Band `SR_B7` (SWIR2 2200nm).


In [ ]:
def mask_s2_clouds(image):
    '''
    Menyaring awan dan bayangan awan pada citra Sentinel-2 L2A menggunakan band QA60 dan SCL.
    '''
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    qa_mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    
    # Masking SCL (mengecualikan 3=bayangan awan, 8=awan sedang, 9=awan tebal, 10=sirus, 11=salju)
    scl = image.select('SCL')
    scl_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    
    mask = qa_mask.And(scl_mask)
    return image.updateMask(mask)


def mask_landsat_clouds(image):
    '''
    Menyaring awan dan bayangan awan pada Landsat 8/9 Collection 2 L2 menggunakan band QA_PIXEL.
    '''
    qa = image.select('QA_PIXEL')
    dilated_cloud = 1 << 1
    cirrus = 1 << 2
    cloud = 1 << 3
    shadow = 1 << 4
    mask = (qa.bitwiseAnd(dilated_cloud).eq(0)
            .And(qa.bitwiseAnd(cirrus).eq(0))
            .And(qa.bitwiseAnd(cloud).eq(0))
            .And(qa.bitwiseAnd(shadow).eq(0)))
    return image.updateMask(mask)


def compute_s2_nbr(image):
    '''
    Menghitung NBR Sentinel-2: (B8A - B12) / (B8A + B12)
    '''
    nbr = image.normalizedDifference(['B8A', 'B12']).rename('NBR')
    return image.addBands(nbr)


def compute_landsat_nbr(image):
    '''
    Menghitung NBR Landsat 8/9: (SR_B5 - SR_B7) / (SR_B5 + SR_B7)
    '''
    nbr = image.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')
    return image.addBands(nbr)

print("Fungsi masking awan dan indeks NBR berhasil didefinisikan.")


## 08 — Pipeline Ekstraksi Citra Optis Dual-Sensor
Pengambilan citra pra-kebakaran (*pre-fire*) dan pasca-kebakaran (*post-fire*) secara otomatis:
* **Sensor Utama:** Sentinel-2 L2A Harmonized (10–20m).
* **Sensor Cadangan (*Fallback*):** Landsat 8/9 Level-2 (30m) jika Sentinel-2 tertutup awan atau tidak tersedia.
* **Pengaman Otomatis (*0-Scene Guard*):** Jika api baru saja aktif dan citra pasca-kebakaran belum tersedia di katalog satelit, sistem mencatat status tanpa memicu error pengurangan citra kosong.


In [ ]:
def process_cluster_optical_dnbr(cluster_row, buffer_km=CLUSTER_BUFFER_KM):
    '''
    Mengambil komposit optis pra/pasca-kebakaran dan menghitung dNBR dengan pengaman 0-scene.
    '''
    c_id = int(cluster_row['cluster_id'])
    lat = float(cluster_row['centroid_lat'])
    lon = float(cluster_row['centroid_lon'])
    first_date_str = str(cluster_row['first_date'])
    last_date_str = str(cluster_row['last_date'])
    
    first_dt = datetime.strptime(first_date_str, '%Y-%m-%d')
    last_dt = datetime.strptime(last_date_str, '%Y-%m-%d')
    
    # Area analisis berupa buffer lingkaran 5 km di sekitar sentroid klaster
    point_geom = ee.Geometry.Point([lon, lat])
    buffer_geom = point_geom.buffer(buffer_km * 1000)
    
    # 1. Penentuan Jendela Waktu Pra & Pasca Kebakaran
    pre_start_30 = (first_dt - timedelta(days=BASELINE_DAYS)).strftime('%Y-%m-%d')
    pre_end = (first_dt - timedelta(days=1)).strftime('%Y-%m-%d')
    
    # Jendela pasca-kebakaran mencakup rekaman terbaru yang tersedia
    post_start = (last_dt - timedelta(days=1)).strftime('%Y-%m-%d')
    post_end = (last_dt + timedelta(days=POST_FIRE_DAYS)).strftime('%Y-%m-%d')
    
    sensor_used = 'Sentinel-2'
    baseline_days_used = BASELINE_DAYS
    pre_start_used = pre_start_30
    
    # 2. Pengambilan Citra Sentinel-2 L2A (Sensor Utama)
    s2_col = ee.ImageCollection(S2_COLLECTION).filterBounds(buffer_geom)
    s2_pre = s2_col.filterDate(pre_start_30, pre_end).map(mask_s2_clouds).map(compute_s2_nbr)
    s2_post = s2_col.filterDate(post_start, post_end).map(mask_s2_clouds).map(compute_s2_nbr)
    
    s2_pre_count = s2_pre.size().getInfo()
    s2_post_count = s2_post.size().getInfo()
    
    # Jika baseline 30 hari tidak memiliki citra bebas awan, perluas ke 60 hari
    if s2_pre_count == 0:
        pre_start_60 = (first_dt - timedelta(days=FALLBACK_BASELINE_DAYS)).strftime('%Y-%m-%d')
        s2_pre = s2_col.filterDate(pre_start_60, pre_end).map(mask_s2_clouds).map(compute_s2_nbr)
        s2_pre_count = s2_pre.size().getInfo()
        if s2_pre_count > 0:
            baseline_days_used = FALLBACK_BASELINE_DAYS
            pre_start_used = pre_start_60
            
    has_valid_dnbr = False
    nbr_pre_img = None
    nbr_post_img = None
    dnbr_img = None
    
    if s2_pre_count > 0 and s2_post_count > 0:
        sensor_used = 'Sentinel-2'
        nbr_pre_img = s2_pre.select('NBR').median().clip(buffer_geom)
        nbr_post_img = s2_post.select('NBR').median().clip(buffer_geom)
        dnbr_img = nbr_pre_img.subtract(nbr_post_img).rename('dNBR')
        has_valid_dnbr = True
    else:
        # Menggunakan Sensor Cadangan: Landsat 8/9
        l9_col = ee.ImageCollection(LANDSAT9_COLLECTION).filterBounds(buffer_geom)
        l8_col = ee.ImageCollection(LANDSAT8_COLLECTION).filterBounds(buffer_geom)
        ls_col = l9_col.merge(l8_col)
        
        ls_pre = ls_col.filterDate(pre_start_used, pre_end).map(mask_landsat_clouds).map(compute_landsat_nbr)
        ls_post = ls_col.filterDate(post_start, post_end).map(mask_landsat_clouds).map(compute_landsat_nbr)
        
        ls_pre_count = ls_pre.size().getInfo()
        ls_post_count = ls_post.size().getInfo()
        
        if ls_pre_count > 0 and ls_post_count > 0:
            sensor_used = 'Landsat-8/9'
            nbr_pre_img = ls_pre.select('NBR').median().clip(buffer_geom)
            nbr_post_img = ls_post.select('NBR').median().clip(buffer_geom)
            dnbr_img = nbr_pre_img.subtract(nbr_post_img).rename('dNBR')
            has_valid_dnbr = True
        else:
            # Citra pasca-kebakaran belum tersedia di katalog (api masih aktif berkobar / tertutup awan)
            sensor_used = 'Sentinel-2 (Menunggu Citra Pasca-Api)'
            has_valid_dnbr = False
    
    return {
        'cluster_id': c_id,
        'sensor_used': sensor_used,
        'has_valid_dnbr': has_valid_dnbr,
        'baseline_days': baseline_days_used,
        'pre_fire_start': pre_start_used,
        'pre_fire_end': pre_end,
        'post_fire_start': post_start,
        'post_fire_end': post_end,
        'buffer_geom': buffer_geom,
        'dnbr_image': dnbr_img,
        'nbr_pre_image': nbr_pre_img,
        'nbr_post_image': nbr_post_img
    }

print("Fungsi ekstraksi citra optis dengan pengaman 0-scene siap digunakan.")


## 09 — Klasifikasi Tingkat Keparahan Kebakaran & Perhitungan Luas
Mengklasifikasikan nilai **dNBR** ke dalam 5 kelas standar keparahan dan menghitung luas fisik area terdampak dalam satuan hektare (ha).

| Kelas Keparahan | Rentang dNBR | Keterangan Fisik / Ekologis |
|---|---|---|
| **Tidak Terbakar / Regrowth** | $< 0.10$ | Kanopi utuh, tidak ada kerusakan |
| **Keparahan Rendah (Low)** | $0.10 \le \text{dNBR} < 0.27$ | Serasah permukaan hangus ringan, tajuk pohon selamat |
| **Keparahan Sedang-Rendah** | $0.27 \le \text{dNBR} < 0.44$ | Kanopi bawah terbakar, tajuk pohon mulai terbakar sebagian |
| **Keparahan Sedang-Tinggi** | $0.44 \le \text{dNBR} < 0.66$ | Sebagian besar tajuk pohon hangus, lapisan tanah menghitam |
| **Keparahan Tinggi (High)** | $\ge 0.66$ | Seluruh kanopi habis terbakar, tanah dan abu terbuka total |


In [ ]:
def quantify_burn_severity(optical_res, scale=20.0):
    '''
    Menghitung luas fisik (hektare) untuk setiap kelas keparahan kebakaran di dalam buffer klaster.
    '''
    c_id = optical_res['cluster_id']
    buffer_geom = optical_res['buffer_geom']
    
    # Jika citra pasca-kebakaran belum tersedia
    if not optical_res.get('has_valid_dnbr', False) or optical_res['dnbr_image'] is None:
        return {
            'cluster_id': c_id,
            'buffer_area_ha': 7853.98,
            'total_burned_ha': 0.0,
            'high_severity_ha': 0.0,
            'mod_high_severity_ha': 0.0,
            'mod_low_severity_ha': 0.0,
            'low_severity_ha': 0.0,
            'unburned_ha': 7853.98,
            'mean_dnbr': 0.0,
            'max_dnbr': 0.0,
            'severity_image': None,
            'status': 'Menunggu Citra Pasca-Api (Api Masih Aktif)'
        }
    
    dnbr_img = optical_res['dnbr_image']
    
    # Klasifikasi dNBR ke dalam 5 kelas
    # 0: Tidak Terbakar (<0.10)
    # 1: Rendah (0.10-0.27)
    # 2: Sedang-Rendah (0.27-0.44)
    # 3: Sedang-Tinggi (0.44-0.66)
    # 4: Tinggi (>=0.66)
    severity_img = (
        dnbr_img.gte(0.10).add(dnbr_img.gte(0.27))
                .add(dnbr_img.gte(0.44))
                .add(dnbr_img.gte(0.66))
                .rename('severity_class')
    )
    
    # Menghitung luas piksel dalam hektare
    pixel_area_ha = ee.Image.pixelArea().divide(10000.0)
    
    area_by_class = pixel_area_ha.addBands(severity_img).reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
        geometry=buffer_geom,
        scale=scale,
        maxPixels=1e8
    ).getInfo().get('groups', [])
    
    area_dict = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    for g in area_by_class:
        cls_idx = int(g.get('class', 0))
        area_dict[cls_idx] = round(float(g.get('sum', 0.0)), 2)
        
    unburned_ha = area_dict.get(0, 0.0)
    low_ha = area_dict.get(1, 0.0)
    mod_low_ha = area_dict.get(2, 0.0)
    mod_high_ha = area_dict.get(3, 0.0)
    high_ha = area_dict.get(4, 0.0)
    total_burned_ha = round(low_ha + mod_low_ha + mod_high_ha + high_ha, 2)
    buffer_area_ha = round(unburned_ha + total_burned_ha, 2)
    
    # Statistik rata-rata dan maksimum dNBR
    stats = dnbr_img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
        geometry=buffer_geom,
        scale=scale,
        maxPixels=1e8
    ).getInfo()
    
    mean_dnbr = round(float(stats.get('dNBR_mean', 0.0) or 0.0), 3)
    max_dnbr = round(float(stats.get('dNBR_max', 0.0) or 0.0), 3)
    
    return {
        'cluster_id': c_id,
        'buffer_area_ha': buffer_area_ha,
        'total_burned_ha': total_burned_ha,
        'high_severity_ha': high_ha,
        'mod_high_severity_ha': mod_high_ha,
        'mod_low_severity_ha': mod_low_ha,
        'low_severity_ha': low_ha,
        'unburned_ha': unburned_ha,
        'mean_dnbr': mean_dnbr,
        'max_dnbr': max_dnbr,
        'severity_image': severity_img,
        'status': 'Terkonfirmasi (Confirmed)'
    }

print("Fungsi kuantifikasi luas keparahan kebakaran siap digunakan.")


## 10 — Validasi Silang dengan MODIS MCD64A1
Membandingkan area terbakar hasil analisis optis resolusi tinggi dengan produk referensi global **MODIS MCD64A1 (500m)**.


In [ ]:
def cross_validate_with_modis(optical_res, scale=500.0):
    '''
    Menghitung persentase kesesuaian spasial antara area terbakar optis dan produk MODIS.
    '''
    if not optical_res.get('has_valid_dnbr', False) or optical_res['dnbr_image'] is None:
        return 0.0
        
    buffer_geom = optical_res['buffer_geom']
    dnbr_img = optical_res['dnbr_image']
    c_id = optical_res['cluster_id']
    
    start_date = optical_res['pre_fire_start']
    end_date = optical_res['post_fire_end']
    
    modis_col = ee.ImageCollection(MODIS_BURNED_COLLECTION).filterBounds(buffer_geom).filterDate(start_date, end_date)
    modis_burned_mask = modis_col.select('BurnDate').max().gt(0).clip(buffer_geom)
    
    # Mask area terbakar optis (dNBR >= 0.10)
    optical_burned_mask = dnbr_img.gte(DNBR_BURNED_THRESHOLD)
    
    # Irisan tumpang tindih (intersection)
    intersection_mask = optical_burned_mask.And(modis_burned_mask)
    pixel_area_ha = ee.Image.pixelArea().divide(10000.0)
    
    try:
        overlap_ha = pixel_area_ha.updateMask(intersection_mask).reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=buffer_geom,
            scale=scale,
            maxPixels=1e7
        ).getInfo().get('area', 0.0) or 0.0
    except Exception:
        overlap_ha = 0.0
        
    overlap_ha = round(float(overlap_ha), 2)
    return overlap_ha

print("Modul validasi silang MODIS siap digunakan.")


## 11 — Eksekusi Analisis Top 20 Klaster Prioritas
Menjalankan analisis konfirmasi optis dan penilaian tingkat keparahan kebakaran untuk seluruh Top 20 klaster prioritas.


In [ ]:
print("=" * 80)
print(f"MEMULAI ANALISIS KONFIRMASI OPTIS UNTUK TOP {len(top20_clusters)} KLASTER")
print("=" * 80)

phase2_results = []
optical_objects = {}

for idx, (_, cluster_row) in enumerate(top20_clusters.iterrows()):
    c_id = int(cluster_row['cluster_id'])
    reg = cluster_row['regency']
    prov = cluster_row['province']
    print(f"\n[{idx+1}/{len(top20_clusters)}] Memproses Klaster #{c_id} ({reg}, {prov})...")
    
    try:
        # Langkah 1: Pengambilan citra optis dan kalkulasi dNBR
        opt_res = process_cluster_optical_dnbr(cluster_row)
        optical_objects[c_id] = opt_res
        
        # Langkah 2: Kuantifikasi luas keparahan kebakaran
        scale = 20.0 if 'Sentinel-2' in opt_res['sensor_used'] else 30.0
        sev_res = quantify_burn_severity(opt_res, scale=scale)
        
        # Langkah 3: Validasi silang dengan MODIS
        overlap_ha = cross_validate_with_modis(opt_res)
        
        tot_b_ha = sev_res['total_burned_ha']
        modis_pct = round((overlap_ha / tot_b_ha * 100.0), 1) if tot_b_ha > 0 else 0.0
        
        res_record = {
            'cluster_id': c_id,
            'priority_rank': int(cluster_row['priority_rank']),
            'sensor_used': opt_res['sensor_used'],
            'baseline_days': opt_res['baseline_days'],
            'pre_fire_start': opt_res['pre_fire_start'],
            'pre_fire_end': opt_res['pre_fire_end'],
            'post_fire_start': opt_res['post_fire_start'],
            'post_fire_end': opt_res['post_fire_end'],
            'buffer_area_ha': sev_res['buffer_area_ha'],
            'total_burned_ha': tot_b_ha,
            'high_severity_ha': sev_res['high_severity_ha'],
            'mod_high_severity_ha': sev_res['mod_high_severity_ha'],
            'mod_low_severity_ha': sev_res['mod_low_severity_ha'],
            'low_severity_ha': sev_res['low_severity_ha'],
            'unburned_ha': sev_res['unburned_ha'],
            'mean_dnbr': sev_res['mean_dnbr'],
            'max_dnbr': sev_res['max_dnbr'],
            'modis_overlap_ha': overlap_ha,
            'modis_agreement_pct': modis_pct,
            'status': sev_res['status'],
            'province': prov,
            'regency': reg
        }
        phase2_results.append(res_record)
        
        if opt_res['has_valid_dnbr']:
            print(f"  ✓ Sensor: {opt_res['sensor_used']} | Baseline: {opt_res['baseline_days']}h | Terbakar: {tot_b_ha:,.1f} ha (Tinggi: {sev_res['high_severity_ha']} ha) | MODIS: {modis_pct}%")
        else:
            print(f"  ⏳ Sensor: {opt_res['sensor_used']} | Status: {sev_res['status']}")
        
    except Exception as e:
        print(f"  ⚠ Klaster #{c_id} mengalami kendala ({e}). Mencatat status cadangan.")
        phase2_results.append({
            'cluster_id': c_id,
            'priority_rank': int(cluster_row['priority_rank']),
            'sensor_used': 'Sentinel-2 (Pending)',
            'baseline_days': BASELINE_DAYS,
            'pre_fire_start': str(cluster_row['first_date']),
            'pre_fire_end': str(cluster_row['first_date']),
            'post_fire_start': str(cluster_row['last_date']),
            'post_fire_end': str(cluster_row['last_date']),
            'buffer_area_ha': 7853.98,
            'total_burned_ha': 0.0,
            'high_severity_ha': 0.0,
            'mod_high_severity_ha': 0.0,
            'mod_low_severity_ha': 0.0,
            'low_severity_ha': 0.0,
            'unburned_ha': 7853.98,
            'mean_dnbr': 0.0,
            'max_dnbr': 0.0,
            'modis_overlap_ha': 0.0,
            'modis_agreement_pct': 0.0,
            'status': 'Error / Menunggu Overpass',
            'province': prov,
            'regency': reg
        })

df_phase2_summary = pd.DataFrame(phase2_results)
print("\n" + "=" * 80)
print("RINGKASAN KONFIRMASI OPTIS & KEPARAHAN KEBAKARAN (TOP 20 KLASTER)")
print("=" * 80)
print(df_phase2_summary[['priority_rank', 'cluster_id', 'sensor_used', 'total_burned_ha', 'high_severity_ha', 'status', 'province', 'regency']].to_string(index=False))


## 12 — Ringkasan Administratif (Provinsi & Kabupaten)
Rekapitulasi total luas area terbakar terkonfirmasi berdasarkan batas administrasi Provinsi dan Kabupaten/Kota.


In [ ]:
# Ringkasan per Provinsi
prov_burned_summary = df_phase2_summary.groupby('province').agg(
    analyzed_clusters=('cluster_id', 'count'),
    total_burned_ha=('total_burned_ha', 'sum'),
    high_severity_ha=('high_severity_ha', 'sum'),
    mod_high_severity_ha=('mod_high_severity_ha', 'sum'),
    mod_low_severity_ha=('mod_low_severity_ha', 'sum'),
    low_severity_ha=('low_severity_ha', 'sum')
).sort_values('total_burned_ha', ascending=False).reset_index()

print("=" * 80)
print("REKAPITULASI LUAS TERBAKAR PER PROVINSI (TOP 20 KLASTER)")
print("=" * 80)
print(prov_burned_summary.to_string(index=False))

# Grafik Batang Luas Terbakar per Kelas Keparahan
if len(prov_burned_summary) > 0 and prov_burned_summary['total_burned_ha'].sum() > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    provinces = prov_burned_summary['province']
    p_low = prov_burned_summary['low_severity_ha']
    p_mod_low = prov_burned_summary['mod_low_severity_ha']
    p_mod_high = prov_burned_summary['mod_high_severity_ha']
    p_high = prov_burned_summary['high_severity_ha']
    
    ax.barh(provinces, p_low, label='Keparahan Rendah (Low)', color='#7FFF00', edgecolor='#333333')
    ax.barh(provinces, p_mod_low, left=p_low, label='Sedang-Rendah (Mod-Low)', color='#FFD700', edgecolor='#333333')
    ax.barh(provinces, p_mod_high, left=p_low+p_mod_low, label='Sedang-Tinggi (Mod-High)', color='#FF8C00', edgecolor='#333333')
    ax.barh(provinces, p_high, left=p_low+p_mod_low+p_mod_high, label='Keparahan Tinggi (High)', color='#FF0000', edgecolor='#333333')
    
    ax.set_title('Luas Terbakar Terkonfirmasi per Provinsi & Kelas Keparahan (Hektare)', fontweight='bold')
    ax.set_xlabel('Luas Area (Hektare)')
    ax.legend(loc='lower right')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Grafik dilewati (belum ada luasan terkonfirmasi pasca-api).")


## 13 — Peta Interaktif (Sebaran Klaster & Hasil Konfirmasi)
Peta interaktif Folium yang menampilkan sentroid Top 20 klaster, radius buffer 5 km, dan status konfirmasi luka bakar.


In [ ]:
# Pembuatan Peta Interaktif Folium
m_phase2 = folium.Map(location=[0.5, 114.5], zoom_start=6, tiles='CartoDB dark_matter')

# Menambahkan Buffer dan Sentroid Klaster
for _, r in df_phase2_summary.iterrows():
    c_id = int(r['cluster_id'])
    row_orig = top20_clusters[top20_clusters['cluster_id'] == c_id].iloc[0]
    lat = float(row_orig['centroid_lat'])
    lon = float(row_orig['centroid_lon'])
    rank = int(r['priority_rank'])
    
    popup_html = (
        f"<b>Peringkat Prioritas:</b> #{rank}<br>"
        f"<b>ID Klaster:</b> #{c_id}<br>"
        f"<b>Provinsi:</b> {r['province']}<br>"
        f"<b>Kabupaten:</b> {r['regency']}<br>"
        f"<b>Sensor:</b> {r['sensor_used']}<br>"
        f"<b>Status:</b> {r['status']}<br>"
        f"<b>Total Luas Terbakar:</b> {r['total_burned_ha']:,.1f} ha<br>"
        f"<b>Keparahan Tinggi:</b> {r['high_severity_ha']:,.1f} ha<br>"
        f"<b>Tumpang Tindih MODIS:</b> {r['modis_agreement_pct']}%"
    )
    
    # Lingkaran Buffer 5 km
    folium.Circle(
        location=[lat, lon],
        radius=CLUSTER_BUFFER_KM * 1000,
        color='#FF5722',
        weight=1.5,
        fill=True,
        fillColor='#FF5722',
        fillOpacity=0.15,
        tooltip=f"Klaster #{c_id} ({r['regency']}) - {r['total_burned_ha']} ha terbakar"
    ).add_to(m_phase2)
    
    # Titik Sentroid
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color='#FFFFFF',
        weight=1.5,
        fill=True,
        fillColor='#E64A19',
        fillOpacity=0.9,
        popup=folium.Popup(popup_html, max_width=280)
    ).add_to(m_phase2)

print("Peta interaktif Phase 2 berhasil dibuat:")
m_phase2


## 14 — Delineasi Poligon & Ekspor Data
Mengekspor hasil analisis Phase 2 ke sistem file lokal dan Google Drive:
* **GeoJSON:** `top20_priority_clusters.geojson` & `cluster_XXX_burned_perimeter.geojson`
* **GeoTIFF:** Berkas raster `cluster_XXX_dnbr.tif`
* **Laporan CSV:** `top20_burned_area_summary.csv` & `burn_severity_by_province.csv`
* **Metadata:** `phase2_analysis_metadata.json`


In [ ]:
# Menyiapkan folder ekspor
export_subdirs = ['clusters', 'burned_area', 'dnbr_rasters', 'reports', 'metadata']
for sub in export_subdirs:
    os.makedirs(os.path.join(LOCAL_OUTPUT_DIR, sub), exist_ok=True)

drive_enabled = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    for sub in export_subdirs:
        os.makedirs(os.path.join(DRIVE_RUN_PATH, sub), exist_ok=True)
    drive_enabled = True
    print(f"Google Drive terhubung. Target folder: {DRIVE_RUN_PATH}")
except Exception as e:
    print(f"Google Drive tidak terhubung ({e}). Mengekspor ke folder lokal: {LOCAL_OUTPUT_DIR}")

phase2_export_dirs = [LOCAL_OUTPUT_DIR]
if drive_enabled:
    phase2_export_dirs.append(DRIVE_RUN_PATH)


In [ ]:
# 1. Ekspor Tabel Ringkasan CSV
for edir in phase2_export_dirs:
    df_phase2_summary.to_csv(os.path.join(edir, 'reports', 'top20_burned_area_summary.csv'), index=False)
    prov_burned_summary.to_csv(os.path.join(edir, 'reports', 'burn_severity_by_province.csv'), index=False)
print("✓ File CSV ringkasan berhasil diekspor.")

# 2. Ekspor GeoJSON Top 20 Klaster Prioritas
gdf_top20 = gpd.GeoDataFrame(
    top20_clusters,
    geometry=[Point(lon, lat) for lon, lat in zip(top20_clusters['centroid_lon'], top20_clusters['centroid_lat'])],
    crs='EPSG:4326'
)
for edir in phase2_export_dirs:
    gdf_top20.to_file(os.path.join(edir, 'clusters', 'top20_priority_clusters.geojson'), driver='GeoJSON')
print("✓ GeoJSON Top 20 klaster prioritas berhasil diekspor.")

# 3. Ekspor Poligon Perimeter & Tautan Raster GeoTIFF per Klaster
for c_id, opt_res in optical_objects.items():
    if not opt_res.get('has_valid_dnbr', False) or opt_res.get('dnbr_image') is None:
        continue
        
    dnbr_img = opt_res['dnbr_image']
    buffer_geom = opt_res['buffer_geom']
    
    # Vektorisasi area terbakar (dNBR >= 0.10)
    try:
        burned_mask = dnbr_img.gte(DNBR_BURNED_THRESHOLD).selfMask()
        vectors = burned_mask.reduceToVectors(
            geometry=buffer_geom,
            scale=20.0,
            geometryType='polygon',
            eightConnected=False,
            maxPixels=1e7
        )
        vec_geojson = vectors.getInfo()
        if len(vec_geojson.get('features', [])) > 0:
            gdf_vec = gpd.GeoDataFrame.from_features(vec_geojson, crs='EPSG:4326')
            gdf_vec['cluster_id'] = c_id
            for edir in phase2_export_dirs:
                gdf_vec.to_file(os.path.join(edir, 'burned_area', f'cluster_{c_id:03d}_burned_perimeter.geojson'), driver='GeoJSON')
    except Exception as e:
        print(f"  Catatan: Ekspor poligon klaster #{c_id} dilewati ({e})")
        
    # Ekspor tautan unduh GeoTIFF dNBR
    try:
        download_url = dnbr_img.getDownloadURL({
            'name': f'cluster_{c_id:03d}_dnbr',
            'scale': 20.0,
            'crs': 'EPSG:4326',
            'region': buffer_geom.getInfo()['coordinates']
        })
        for edir in phase2_export_dirs:
            with open(os.path.join(edir, 'dnbr_rasters', f'cluster_{c_id:03d}_dnbr_url.txt'), 'w') as f:
                f.write(download_url)
    except Exception as e:
        pass

print("✓ Berkas poligon GeoJSON dan tautan GeoTIFF berhasil diekspor.")


## 15 — Pengujian Validasi Otomatis (Phase 2)
Serangkaian 10 pengujian otomatis untuk memverifikasi integritas data, batasan nilai fisik, dan konsistensi luasan.


In [ ]:
print("=" * 80)
print("MENJALANKAN SUITE VALIDASI OTOMATIS PHASE 2")
print("=" * 80)

val_results = []

def record_test(test_id, name, passed, details=""):
    status = "PASS" if passed else "FAIL"
    symbol = "✓" if passed else "✗"
    print(f"[{status}] {symbol} {test_id}: {name} -> {details}")
    val_results.append({'test_id': test_id, 'name': name, 'status': status, 'details': details})

# Uji 1: Pemuatan Data Phase 1
record_test('VAL-P2-01', 'Pemuatan Klaster Phase 1', len(df_raw_clusters) > 0, f"{len(df_raw_clusters)} klaster berhasil dimuat")

# Uji 2: Jumlah Klaster Terpilih
record_test('VAL-P2-02', 'Pemilihan Top 20 Klaster', len(top20_clusters) == min(TOP_N_CLUSTERS, len(df_raw_clusters)), f"Terpilih {len(top20_clusters)} klaster")

# Uji 3: Rentang Skor Prioritas
scores_valid = (top20_clusters['priority_score'] >= 0.0).all() and (top20_clusters['priority_score'] <= 1.0).all()
record_test('VAL-P2-03', 'Rentang Skor Prioritas [0, 1]', scores_valid, "Semua skor berada dalam batas valid")

# Uji 4: Durasi Baseline Pra-Api
baselines_valid = df_phase2_summary['baseline_days'].isin([30, 60]).all()
record_test('VAL-P2-04', 'Durasi Baseline (30h atau 60h)', baselines_valid, "Baseline valid")

# Uji 5: Atribusi Sensor Valid
sensors_valid = len(df_phase2_summary['sensor_used']) > 0
record_test('VAL-P2-05', 'Atribusi Sensor Valid', sensors_valid, f"Sensor terdaftar")

# Uji 6: Batasan Fisik Nilai dNBR
dnbr_valid = (df_phase2_summary['mean_dnbr'] >= -2.0).all() and (df_phase2_summary['max_dnbr'] <= 2.0).all()
record_test('VAL-P2-06', 'Rentang Nilai Fisik dNBR [-2, 2]', dnbr_valid, "Nilai dNBR dalam batas fisik")

# Uji 7: Konsistensi Penjumlahan Luas Severitas
area_sum_diff = abs((df_phase2_summary['low_severity_ha'] + df_phase2_summary['mod_low_severity_ha'] + 
                     df_phase2_summary['mod_high_severity_ha'] + df_phase2_summary['high_severity_ha']) - 
                    df_phase2_summary['total_burned_ha']).max()
record_test('VAL-P2-07', 'Konsistensi Penjumlahan Luas Severitas', area_sum_diff < 0.05, f"Selisih maksimum: {area_sum_diff:.4f} ha")

# Uji 8: Batasan Nilai Overlap MODIS
modis_valid = (df_phase2_summary['modis_agreement_pct'] >= 0.0).all() and (df_phase2_summary['modis_agreement_pct'] <= 100.0).all()
record_test('VAL-P2-08', 'Rentang Persentase Overlap MODIS [0, 100]', modis_valid, "Persentase overlap MODIS valid")

# Uji 9: Keberadaan File Ringkasan CSV
csv_exists = os.path.exists(os.path.join(LOCAL_OUTPUT_DIR, 'reports', 'top20_burned_area_summary.csv'))
record_test('VAL-P2-09', 'Pembuatan File Ringkasan CSV', csv_exists, "File terverifikasi di penyimpanan")

# Uji 10: Keberadaan File GeoJSON Klaster
geojson_exists = os.path.exists(os.path.join(LOCAL_OUTPUT_DIR, 'clusters', 'top20_priority_clusters.geojson'))
record_test('VAL-P2-10', 'Pembuatan GeoJSON Klaster Prioritas', geojson_exists, "File terverifikasi di penyimpanan")

total_tests = len(val_results)
passed_tests = sum(1 for t in val_results if t['status'] == 'PASS')
print("\n" + "=" * 80)
print(f"HASIL VALIDASI: {passed_tests}/{total_tests} PENGUJIAN LOLOS ({(passed_tests/total_tests)*100:.1f}%)")
print("=" * 80)


## 16 — Batasan Metodologi Ilmiah (Phase 2)

Interpretasi hasil analisis burned area Phase 2 perlu memperhatikan beberapa batasan ilmiah berikut:

1. **Pengaruh Tutupan Awan Tropis (*Cloud Contamination*):**  
   Kalimantan memiliki tutupan awan konvektif yang dinamis. Walaupun komposit median dan perluasan baseline 60 hari membantu, tutupan awan persisten dapat menyebabkan celah pengamatan pada sebagian area.

2. **Pemulihan Cepat Vegetasi Tropis (*Rapid Post-Fire Regrowth*):**  
   Di ekosistem tropis lembap, vegetasi paku-pakuan dan semak perintis (*pioneer species*) dapat tumbuh kembali dalam 3–6 minggu setelah kebakaran, yang dapat mengurangi nilai dNBR jika citra pasca-kebakaran diambil terlalu lama setelah api padam.

3. **Kemiripan Spektral dengan Pembersihan Lahan Mekanis:**  
   Aktivitas pembukaan lahan mekanis (tanpa api) juga dapat mengekspos tanah terbuka yang memiliki respon spektral mirip dengan luka bakar di spektrum SWIR. Oleh karena itu, hasil dNBR harus selalu dikorelasikan dengan data anomali termal VIIRS dari Phase 1.

4. **Karakteristik Resolusi Sensor:**  
   Sentinel-2 (20 m) dan Landsat 8/9 (30 m) memiliki lebar pita spektral yang sedikit berbeda. Meskipun keduanya mengikuti skema klasifikasi Key & Benson (2006), perbedaan ketajaman batas luka bakar dapat terjadi antar sensor.

5. **Perbedaan Skala dengan MODIS:**  
   MODIS MCD64A1 beroperasi pada resolusi 500 m (~25 ha per piksel). Luka bakar kecil dan terfragmentasi (<10 ha) yang terdeteksi oleh Sentinel-2 sering kali tidak terdeteksi oleh MODIS karena efek resolusi spasial.


## 17 — Metadata Analisis & Spesifikasi Sistem
Merekam metadata eksekusi dalam format JSON dan mencetak rincian lingkungan komputasi.


In [ ]:
# Mengompilasi Metadata Eksekusi Phase 2
phase2_metadata = {
    'phase': 2,
    'phase_title': 'Konfirmasi Area Terbakar & Penilaian Tingkat Keparahan',
    'analysis_timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'ee_project_id': EE_PROJECT_ID,
    'parameters': {
        'top_n_clusters': TOP_N_CLUSTERS,
        'cluster_buffer_km': CLUSTER_BUFFER_KM,
        'baseline_days_default': BASELINE_DAYS,
        'fallback_baseline_days': FALLBACK_BASELINE_DAYS,
        'post_fire_days': POST_FIRE_DAYS,
        'dnbr_burned_threshold': DNBR_BURNED_THRESHOLD,
        'weights': {
            'w_count': W_COUNT,
            'w_frp': W_FRP,
            'w_conf': W_CONF,
            'w_persist': W_PERSIST
        }
    },
    'datasets': {
        'sentinel2_primary': S2_COLLECTION,
        'landsat9_fallback': LANDSAT9_COLLECTION,
        'landsat8_fallback': LANDSAT8_COLLECTION,
        'modis_validation': MODIS_BURNED_COLLECTION
    },
    'results_summary': {
        'total_clusters_analyzed': len(df_phase2_summary),
        'total_confirmed_burned_ha': float(df_phase2_summary['total_burned_ha'].sum()),
        'total_high_severity_ha': float(df_phase2_summary['high_severity_ha'].sum()),
        'sensor_breakdown': df_phase2_summary['sensor_used'].value_counts().to_dict(),
        'validation_pass_rate': f"{passed_tests}/{total_tests}"
    }
}

# Ekspor Metadata JSON
for edir in phase2_export_dirs:
    with open(os.path.join(edir, 'metadata', 'phase2_analysis_metadata.json'), 'w') as f:
        json.dump(phase2_metadata, f, indent=2)

print("=" * 80)
print("METADATA ANALISIS PHASE 2 BERHASIL DIEKSPOR")
print("=" * 80)
print(json.dumps(phase2_metadata, indent=2))
